# 🎙️ tech-history 음성 생산 v3 — 캐글판 (코랩 GPU 한도 소진 시 대체 경로)

**이 노트북이 맞는지 확인:** 세 번째 셀 실행 시 `[v3-kaggle] 숫자 한글 변환 켜짐` 출력.
**현재 설정: 2편(EPISODE="02") 전체 15조각 생산.**

## 최초 1회 준비 (캐글 계정)

1. kaggle.com 가입 → 프로필 → Settings → **Phone verification** (전화 인증 — GPU·인터넷 사용에 필수)
2. 이 노트북 열기: kaggle.com → 왼쪽 **Create → New Notebook** → 메뉴 **File → Import Notebook** → 이 파일 업로드

## 실행 전 설정 (오른쪽 패널 아래 Session options — 순서 중요!)

1. **Accelerator: GPU T4 x2** (또는 P100) 선택
2. **Internet: ON** (대본·모델 다운로드에 필수)
3. **ref.wav 업로드**: 오른쪽 패널 **+ Add Input → Upload** →
   `ref.wav` 파일 선택(PC 위치: `video/output/ref.wav`), 데이터셋 이름 아무거나(예: haesollo-ref),
   Private 확인 후 Create → 목록에 붙을 때까지 잠시 대기 (한 번 올려두면 다음부터 재사용)

## 실행 순서 (코랩과 동일)

1. **1번 셀만** 실행 (설치 2~3분)
2. 상단 메뉴 **Run → Restart & clear cell outputs** (필수)
3. **Run All** (1번 셀 재실행은 금방 지나갑니다)
4. 끝나면 마지막 셀이 zip 경로를 출력 → 오른쪽 패널 **Output**(새로고침) → `voice_02_fix.zip` 옆 ⋮ → **Download**

## 다른 편/부분 재생산으로 바꾸려면

- 편 번호: 1번 셀 `EPISODE = "02"` 수정
- 특정 조각만: 4번 셀 `ONLY = [11]` 처럼 번호 목록 지정 (빈 `[]` = 전부)

In [ ]:
# ── 1번 셀: 설치 ──
EPISODE = "02"  # 생산할 편 번호
!pip install -q chatterbox-tts requests
!pip uninstall -y -q torchvision  # 미사용 부품 — 구버전 torch와 충돌하므로 제거
print("설치 완료 — 이제 메뉴에서 [Run → Restart & clear cell outputs] 후 [Run All] 하세요")

In [ ]:
# ── 2번 셀: ref.wav 탐색 (업로드한 데이터셋에서 작업 폴더로 복사) ──
import glob, shutil, os
found = glob.glob("/kaggle/input/**/ref.wav", recursive=True)
if found:
    shutil.copyfile(found[0], "ref.wav")
    print("ref.wav 확보:", found[0])
else:
    print("⚠️ ref.wav 없음 — 기본 목소리로 생산됩니다. 육성 적용이 목적이면")
    print("   오른쪽 + Add Input > Upload 로 ref.wav를 올리고 이 셀부터 다시 실행하세요.")

In [ ]:
# ── 3번 셀: 대본·모델 로드 ──
print("[v3-kaggle] 숫자 한글 변환 켜짐")  # 이식판 확인 표식
import json, os, re, shutil, requests, torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

url = f"https://raw.githubusercontent.com/nous-zero/tech-history/main/video/scripts/{EPISODE}.json"
script = requests.get(url).json()
print("대본:", script["title"], "/ 문단", len(script["segments"]))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치:", device)
assert device == "cuda", "GPU가 안 잡혔습니다 — 오른쪽 Notebook options에서 Accelerator를 GPU로 바꾸세요"
model = ChatterboxMultilingualTTS.from_pretrained(device=device)

In [ ]:
# ── 4번 셀: 음성 생산 (콜랩 v3와 동일 로직) ──
# 재생산할 세그먼트 번호 목록. 빈 목록 [] = 전부 생산.
# (1편 부분 재생산이 필요하면 1번 셀 EPISODE="01" + 여기에 번호 목록. 예: ONLY = [11])
ONLY = []

# --- 숫자 → 한글 발음 변환 (TTS 입력 전용 — 화면 자막은 원문 숫자 유지) ---
_SINO = "영일이삼사오육칠팔구"

def _sino(n):
    n = int(n)
    if n == 0:
        return "영"
    out = ""
    for val, name in ((10000, "만"), (1000, "천"), (100, "백"), (10, "십"), (1, "")):
        d, n = n // val, n % val
        if d:
            out += ("" if d == 1 and name else _SINO[d]) + name
    return out

_MONTH = {6: "유", 10: "시"}  # 6월=유월, 10월=시월

def normalize_numbers(t):
    t = re.sub(r"(\d+)월", lambda m: (_MONTH.get(int(m.group(1))) or _sino(m.group(1))) + "월", t)
    return re.sub(r"\d+", lambda m: _sino(m.group(0)), t)

REF = "ref.wav" if os.path.exists("ref.wav") else None  # 육성 복제용(선택)
if REF:
    # 주의: ONLY 부분 재생산은 기존 조각들도 같은 ref로 만들어졌을 때만 사용
    # (다른 목소리와 섞이면 안 됨). 처음 생산이면 ONLY = [] 로 전체 생산할 것.
    print("ref.wav 감지 — 육성 복제 모드,", f"부분 재생산 {ONLY}" if ONLY else "전체 재생산")
out_dir = f"voice_{EPISODE}_fix"
shutil.rmtree(out_dir, ignore_errors=True)  # 이전 실행 잔존 파일 제거 — 산출물은 이번 생산분만
os.makedirs(out_dir)
for seg in script["segments"]:
    if ONLY and seg["id"] not in ONLY:
        continue
    text = normalize_numbers(seg["text"])
    print(f"seg{seg['id']:03d} 읽을 문장: {text}")
    kwargs = {"language_id": "ko"}
    if REF:
        kwargs["audio_prompt_path"] = REF
    limit = 0.25 * len(text) + 5  # 비정상 길이(환각 반복) 감시선
    for attempt in range(3):
        wav = model.generate(text, **kwargs)
        sec = wav.shape[-1] / model.sr
        if sec <= limit:
            break
        print(f"  {sec:.1f}초 — 비정상(기준 {limit:.0f}초), 재시도 {attempt + 1}/3")
    torchaudio.save(os.path.join(out_dir, f"seg{seg['id']:03d}.wav"), wav, model.sr)
    print(f"seg{seg['id']:03d} 완료 ({sec:.1f}초)")
print("합성 완료:", "전체" if not ONLY else f"세그먼트 {ONLY}")

In [ ]:
# ── 5번 셀: 압축 → 오른쪽 Output 패널에서 다운로드 ──
import shutil
zip_path = shutil.make_archive(f"voice_{EPISODE}_fix", "zip", f"voice_{EPISODE}_fix")
print("완성:", zip_path)
print("다운로드: 오른쪽 패널 Output 새로고침 → 파일 옆 ⋮ → Download")
from IPython.display import FileLink
FileLink(os.path.basename(zip_path))